# 7.4 · Árboles B — Retos

En esta práctica trabajaremos con los conceptos básicos de un **árbol B**: búsqueda, inserción y división de nodos. Los ejercicios de borrado se resolverán gráficamente, sin implementar `remove()`.


## 📌 Convención utilizada

Trabajaremos principalmente con **Árboles B de orden 4**:

- máximo **4 hijos** por nodo;
- máximo **3 claves** por nodo;
- salvo la raíz (que puede ser nula), los nodos deben conservar al menos **1 clave**.

Representaremos gráficamente un nodo escribiendo sus claves entre corchetes:

```text
[10 | 20]
```

y sus hijos debajo:

```text
          [10 | 20]
        /     |      \
      [5]   [15]   [25 | 30]
```


## ¿Cómo trabajar?

Ejecuta primero la celda que contiene las clases. Resuelve cada reto **completando los `TODO` de esa misma celda** y vuelve a ejecutarla antes de ejecutar sus pruebas. Los retos de programación se realizan progresivamente; los retos gráficos no requieren escribir código.

Los valores de las claves no se repiten.


# Reto 1 · Implementa `BTreeNode` y `BTree`

**Objetivo:** Completa los constructores de las dos clases.

Cada `BTreeNode` tendrá `keys` (claves), `children` (hijos), `parent` (padre) e `is_leaf` (si es hoja). `BTree` tendrá `root` (raíz) y `m` (orden del árbol).

**Tiempo estimado:** 10 minutos · **Dificultad:** Fácil

**Pista:** Un árbol recién creado está vacío: `root` debe ser `None`. El orden `m` debe ser un entero mayor o igual que 3.


In [ ]:
class BTreeNode:
    def __init__(self, is_leaf=True):
        # TODO (Reto 1)
        pass

    def insert_key(self, key):
        # TODO (Reto 2)
        pass


class BTree:
    def __init__(self, m):
        # TODO (Reto 1)
        pass

    def search(self, key):
        # TODO (Reto 3)
        pass

    def _search_recursive(self, node, key):
        # TODO (Reto 3)
        pass

    def search_leaf(self, key):
        # TODO (Reto 4)
        pass

    def insert(self, key):
        # TODO (Reto 5)
        pass

    def _split(self, node):
        # TODO (Reto 5)
        pass


Ejecuta la siguiente celda para comprobar tu solución.


In [ ]:
node = BTreeNode()
assert node.keys == []
assert node.children == []
assert node.parent is None
assert node.is_leaf is True

tree = BTree(4)
assert tree.root is None
assert tree.m == 4

try:
    BTree(2)
    assert False, "Debe rechazarse un orden menor que 3"
except ValueError:
    pass

print("Pruebas superadas para los constructores.")


Pruebas superadas


# Reto 2 · Inserta una clave en un nodo

**Objetivo:** Implementa `BTreeNode.insert_key(key)` para insertar una clave en `self.keys` manteniendo la lista ordenada. No debe insertar duplicados.

Devuelve `True` si se ha insertado la clave y `False` si ya existía.

**Tiempo estimado:** 8 minutos · **Dificultad:** Fácil

**Pista:** Localiza la primera posición cuya clave sea mayor que la clave nueva.

In [ ]:
node = BTreeNode()
node.keys = [10, 20, 40]
assert node.insert_key(30) is True
assert node.keys == [10, 20, 30, 40]
assert node.insert_key(5) is True
assert node.keys == [5, 10, 20, 30, 40]
assert node.insert_key(20) is False
assert node.keys == [5, 10, 20, 30, 40]
print("Pruebas superadas para insert_key().")

# 🏆 Reto 3. Implementa `search`

🎯 **Objetivo:** Implementa la búsqueda recursiva en un Árbol B.

```python
search(key)
_search_recursive(node, key)
```

`search` debe devolver `(node, i)` si encuentra la clave, donde `i` es su posición dentro de `node.keys`. Si no existe, devuelve `None`.

🕒 **Tiempo estimado:** 15 minutos  
🟢 **Dificultad:** Fácil

### 💡 Pista
Dentro de cada nodo:

1. busca la primera posición cuya clave sea mayor o igual que `key`;
2. si coincide, has terminado;
3. si el nodo es hoja, la clave no existe;
4. en otro caso, continúa por `children[i]`.


In [ ]:
t = BTree(4)

root = BTreeNode(False)
root.keys = [30, 70]

left = BTreeNode()
left.keys = [10, 20]

middle = BTreeNode()
middle.keys = [40, 60]

right = BTreeNode()
right.keys = [80, 90]

root.children = [left, middle, right]
for child in root.children:
    child.parent = root

t.root = root

node, pos = t.search(60)
assert node is middle and pos == 1

node, pos = t.search(30)
assert node is root and pos == 0

assert t.search(55) is None
print("Pruebas superadas")


# 🏆 Reto 4. Localiza la hoja correcta para insertar

🎯 **Objetivo:** Implementa:

```python
def search_leaf(self, key)
```

Debe devolver el nodo hoja en el que debería insertarse `key`.

🕒 **Tiempo estimado:** 10 minutos  
🟢 **Dificultad:** Fácil

### 💡 Pista
El procedimiento para elegir el hijo es prácticamente el mismo que en `search`, pero ahora continuamos hasta llegar a una hoja.


In [ ]:
assert t.search_leaf(5) is left
assert t.search_leaf(55) is middle
assert t.search_leaf(95) is right
print("Pruebas superadas")


# Reto 5 · Implementa `insert` con división y propagación

**Objetivo:** Completa `insert(key)` y su método auxiliar `_split(node)`.

La inserción debe:

1. Crear la raíz si el árbol está vacío.
2. Evitar duplicados.
3. Localizar la hoja adecuada e insertar la clave en orden.
4. Dividir un nodo si supera el máximo de `m - 1` claves.
5. Promocionar la mediana al padre y continuar hacia arriba si este también desborda.
6. Crear una nueva raíz si se divide la raíz anterior.

`insert(key)` devuelve `True` si inserta la clave y `False` si ya existía.

**Tiempo estimado:** 25 minutos · **Dificultad:** Intermedia

### Pista

Haz que `_split(node)` resuelva **una única división** y devuelva el padre al que se promociona la mediana. Si crea una nueva raíz, puede devolver `None`. Así, `insert()` puede utilizar un `while` para propagar las divisiones necesarias.


In [ ]:
# Inserciones sin split
simple = BTree(4)
for value in [10, 20, 30]:
    assert simple.insert(value) is True
assert simple.root.keys == [10, 20, 30]

# Split de la raíz
assert simple.insert(40) is True
assert simple.root.keys == [30]
assert simple.root.children[0].keys == [10, 20]
assert simple.root.children[1].keys == [40]

# Más inserciones y búsqueda
for value in [5, 15, 25, 35, 45, 50]:
    assert simple.insert(value) is True

for value in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]:
    assert simple.search(value) is not None

# No se permiten duplicados
assert simple.insert(30) is False
print("Pruebas superadas")


In [ ]:
# Casos adicionales: split propagado, ausencia de duplicados y referencias a padres

def check_tree_structure(tree):
    if tree.root is None:
        return []
    leaf_depths = set()
    collected = []

    def visit(node, parent, lower, upper, depth):
        assert node.parent is parent
        assert node.keys == sorted(node.keys)
        assert len(node.keys) == len(set(node.keys))
        assert len(node.keys) <= tree.m - 1
        if parent is not None:
            assert len(node.keys) >= (tree.m + 1) // 2 - 1
        for key in node.keys:
            assert (lower is None or lower < key)
            assert (upper is None or key < upper)
        if node.is_leaf:
            assert node.children == []
            leaf_depths.add(depth)
            collected.extend(node.keys)
        else:
            assert len(node.children) == len(node.keys) + 1
            bounds = [lower] + node.keys + [upper]
            for i, child in enumerate(node.children):
                visit(child, node, bounds[i], bounds[i+1], depth+1)

    visit(tree.root, None, None, None, 0)
    assert len(leaf_depths) == 1
    return collected

for order in [3, 4, 5, 6]:
    tree = BTree(order)
    values = list(range(1, 81))
    for value in values:
        assert tree.insert(value) is True
    assert all(tree.search(value) is not None for value in values)
    assert tree.search(100) is None
    assert tree.insert(40) is False
    check_tree_structure(tree)
print("Pruebas adicionales superadas para insert() y _split().")

# 🏆 Reto 6. `Insert` gráfico: división de una hoja

🎯 **Objetivo:** Resuelve gráficamente una inserción que provoca un `split`.

Tenemos un Árbol B de orden 4:

```text
          [10 | 20]
        /     |      \
      [5] [12|15|18] [25|30]
```

Inserta la clave **17**.

### ✏️ En tu solución dibuja
1. La hoja después de insertar temporalmente 17 en orden.
2. La división de la hoja.
3. La clave que se promociona al padre.
4. El árbol final.

🕒 **Tiempo estimado:** 10 minutos  
🟢 **Dificultad:** Fácil


# 🏆 Reto 7. `Remove` gráfico: fusión y propagación hasta la raíz

🎯 **Objetivo:** Resolver un borrado en el que una fusión provoca que el **padre también quede deficitario**, por lo que el reajuste debe propagarse un nivel hacia arriba.

Árbol B de orden 4:

```text
                         [50]
                    /            \
                  [20]            [70]
                /      \        /      \
              [10]     [30]   [60]   [80 | 90]
```

Elimina la clave **10**.

### ✏️ Dibuja y explica

1. El estado inmediatamente después de eliminar 10.
2. La fusión necesaria en el nivel inferior.
3. Por qué el nodo interno `[20]` queda deficitario.
4. Cómo se reajusta ahora el nivel superior.
5. El árbol final.

🕒 **Tiempo estimado:** 15 minutos  
🟡 **Dificultad:** Intermedia


# 🏆 Reto 8. `Remove` gráfico: préstamo en el nivel superior tras una fusión

🎯 **Objetivo:** Resolver un borrado en el que el primer reajuste es una **fusión**, pero la propagación al padre se resuelve después mediante **préstamo/rotación**.

Árbol B de orden 4:

```text
                           [50]
                     /              \
                   [20]            [70 | 90]
                 /      \         /    |     \
               [10]     [30]    [60] [80] [100]
```

Elimina la clave **10**.

### ✏️ Dibuja y explica

1. La eliminación de 10.
2. La fusión en el nivel de las hojas.
3. El déficit que aparece en el nodo interno izquierdo.
4. El préstamo desde el hermano interno derecho a través de la raíz.
5. El árbol final.

🕒 **Tiempo estimado:** 18 minutos  
🔴 **Dificultad:** Avanzada


# 🏆 Reto 9. `Remove` gráfico: clave interna y propagación

🎯 **Objetivo:** Combinar el borrado de una **clave interna** con un reajuste que se propaga hacia arriba.

Árbol B de orden 4:

```text
                         [50]
                    /            \
                  [20]            [70]
                /      \        /      \
              [10]     [30]   [60]   [80 | 90]
```

Elimina la clave interna **20** utilizando su **sucesor inmediato**.

### ✏️ Dibuja y explica

1. Localiza el sucesor de 20.
2. Sustituye 20 por el sucesor.
3. Elimina el sucesor de su hoja.
4. Resuelve el déficit de esa hoja.
5. Continúa el reajuste si el nodo padre queda deficitario.
6. Dibuja el árbol final.

🕒 **Tiempo estimado:** 20 minutos  
🔴 **Dificultad:** Avanzada


## Fin de la práctica

Has trabajado la búsqueda y la inserción en árboles B, incluidas la división de nodos y la propagación de medianas. Los últimos retos permiten razonar gráficamente sobre el borrado, las fusiones y los préstamos entre nodos.